# Net-Chat Local Training Notebook
Fine-tune **Qwen2.5-1.5B-Instruct** ด้วย LoRA บน CPU

| รายการ | ค่า |
|--------|-----|
| Base Model | Qwen/Qwen2.5-1.5B-Instruct |
| Method | LoRA (r=16) |
| Dataset | data/combined.jsonl.gz (235,998 pairs) |
| Hardware | CPU (ไม่มี GPU) |

> **หมายเหตุ:** CPU-only จึงใช้ `--samples` จำกัดจำนวน เพื่อให้เสร็จเร็ว

## Cell 1: ตรวจสอบ Environment

In [ ]:
import torch
import sys, os

print(f"Python : {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"Device : {'CUDA - ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# เปลี่ยน working dir ไปที่โปรเจกต์
os.chdir('/home/jcyber/Project-2')
print(f"CWD    : {os.getcwd()}")
print(f"Files  : {os.listdir('.')}")

## Cell 2: แตกไฟล์ dataset (.gz → .jsonl)

In [ ]:
import gzip, shutil

gz_path   = 'data/combined.jsonl.gz'
jsonl_path = 'data/combined.jsonl'

if not os.path.exists(jsonl_path):
    print(f"แตกไฟล์ {gz_path} ...")
    with gzip.open(gz_path, 'rb') as f_in, open(jsonl_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    print(f"แตกไฟล์เสร็จ → {jsonl_path}")
else:
    print(f"มีไฟล์ {jsonl_path} แล้ว ข้ามขั้นตอนนี้")

size_mb = os.path.getsize(jsonl_path) / 1024 / 1024
print(f"ขนาดไฟล์: {size_mb:.1f} MB")

## Cell 3: ดูตัวอย่าง dataset

In [ ]:
import json

samples = []
with open('data/combined.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        samples.append(json.loads(line.strip()))

for i, s in enumerate(samples, 1):
    print(f"--- Sample {i} ---")
    q = s.get('instruction', s.get('question', ''))
    a = s.get('output', s.get('answer', ''))
    print(f"Q: {q[:120]}")
    print(f"A: {a[:120]}")
    print()

## Cell 4: โหลด Model + LoRA Config

จะ download Qwen2.5-1.5B-Instruct (~3GB) จาก HuggingFace ครั้งแรกจะนานหน่อย

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

print(f"กำลังโหลด tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"กำลังโหลด model (อาจใช้เวลานาน)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,   # CPU ต้องใช้ float32
    trust_remote_code=True,
    device_map='cpu'
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("โหลด model + LoRA เสร็จแล้ว")

## Cell 5: เตรียม Dataset

ตั้งค่า `NUM_SAMPLES` เพื่อควบคุมความเร็ว:
- `500` → ทดสอบเร็ว (~15-30 นาที CPU)
- `2000` → ผลดีขึ้น (~1-2 ชั่วโมง CPU)
- `0` → ใช้ทั้งหมด 235,998 pairs (หลายวัน บน CPU)

In [ ]:
from datasets import Dataset

NUM_SAMPLES = 500   # <-- เปลี่ยนตรงนี้
MAX_LENGTH  = 512

def format_prompt(example):
    instruction = example.get('instruction', example.get('question', ''))
    output      = example.get('output', example.get('answer', ''))
    return f"<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"

def tokenize_fn(example):
    text   = format_prompt(example)
    tokens = tokenizer(text, truncation=True, max_length=MAX_LENGTH, padding='max_length')
    tokens['labels'] = tokens['input_ids'].copy()
    return tokens

records = []
with open('data/combined.jsonl', 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue
        if NUM_SAMPLES > 0 and len(records) >= NUM_SAMPLES:
            break

print(f"โหลด {len(records):,} samples")
dataset   = Dataset.from_list(records)
tokenized = dataset.map(tokenize_fn, remove_columns=dataset.column_names, num_proc=4)
print(f"Tokenize เสร็จ: {len(tokenized)} samples พร้อม train")

## Cell 6: Train Model

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

OUTPUT_DIR = 'model/lora-adapter'
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,      # CPU ใช้ batch เล็กๆ
    gradient_accumulation_steps=8,      # สะสม gradient แทน
    save_steps=100,
    logging_steps=20,
    learning_rate=2e-4,
    fp16=False,                         # CPU ไม่รองรับ fp16
    bf16=False,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    dataloader_num_workers=0,           # CPU ใช้ 0 เพื่อความเสถียร
    report_to='none',
    save_total_limit=2,
    load_best_model_at_end=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
)

print("เริ่ม fine-tune... (CPU จะช้ากว่า GPU มาก)")
trainer.train()

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"บันทึก LoRA adapter → {OUTPUT_DIR}")

## Cell 7: ทดสอบ Model หลัง Train

In [ ]:
model.eval()

question = "What is SNMP OID?"
prompt   = f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"
inputs   = tokenizer(prompt, return_tensors='pt')

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

answer = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"Q: {question}")
print(f"A: {answer}")

## Cell 8: Merge LoRA + Export (ถ้าต้องการใช้กับ Ollama)

ขั้นตอนนี้รัน `merge_and_export.py` เพื่อรวม weights และ export เป็น GGUF

In [ ]:
# Merge LoRA กับ base model
!python scripts/merge_and_export.py

In [ ]:
# (ถ้าต้องการ GGUF สำหรับ Ollama — ต้องมี llama.cpp ก่อน)
# !git clone https://github.com/ggerganov/llama.cpp
# !pip install -r llama.cpp/requirements.txt
# !python scripts/merge_and_export.py --gguf --llama-cpp llama.cpp